# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: K-Means clustering.** My lane (Lane 3, structured content archetype clustering) doesn't
have an observed target -- the point is grouping pages that behave alike, not predicting a label.
That rules out logistic regression, decision trees, and boosting, which all need something to
predict. K-Means is the standard fit for "grouping items," and I pick k with silhouette instead of
guessing, then read what's actually inside each cluster before naming anything.

To still compare against my Week-4 baseline on the same metric, I score test pages by their
cluster's observed decline rate (learned on train only) and rank by that -- same precision@K
comparison as a supervised model would get, without ever fitting K-Means on the label itself.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.copy()
test_df = test_df.copy()
print("train:", train_df.shape, "test:", test_df.shape)

shared_clients = set(train_df["client_id"]) & set(test_df["client_id"])
print("clients in both train and test:", len(shared_clients), "of", df["client_id"].nunique())

train: (24000, 44) test: (6000, 44)
clients in both train and test: 31 of 32


**Split: plain random 80/20, not grouped.** This is a single 90-day snapshot -- no dates to hold
out -- so time-aware isn't an option. Grouped-by-client would be the honest choice, but every one
of the 32 clients ends up on both sides of a random split, which means the clusters could be
learning client habits, not page archetypes. I'm leaving it naive on purpose: ML-09 asks me to
re-run this week's model under a grouped split and show the before/after, and that comparison only
means something if this notebook doesn't already do it.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# is_declining_label is for EVALUATION ONLY, never a clustering input.
# trend_direction / trend_pct never enter the feature list below.
for frame in (train_df, test_df):
    frame["is_declining_label"] = (frame["trend_direction"] == "down").astype(int)

print("base rate, whole dataset:", df["trend_direction"].eq("down").mean().round(3))
print("base rate, train:", train_df["is_declining_label"].mean().round(3))
print("base rate, test:", test_df["is_declining_label"].mean().round(3))

base rate, whole dataset: 0.542
base rate, train: 0.541
base rate, test: 0.545


In [4]:
NUMERIC_FEATURES = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "char_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

# leakage check up front: none of the trend/label/future-window columns belong in these lists
banned = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"}
print("banned columns touched:", banned & set(NUMERIC_FEATURES + CATEGORICAL_FEATURES))

def build_features(frame, columns=None):
    X = frame[NUMERIC_FEATURES].copy()
    # a few columns go missing together (feedly articles carry no keyword data at all) --
    # flag it instead of letting a silent fillna(0) pretend "no keyword data" means "zero demand"
    for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
        X[f"has_{col}"] = X[col].notna().astype(int)
    X = X.fillna(0)
    cats = pd.get_dummies(frame[CATEGORICAL_FEATURES].fillna("unknown"), prefix=CATEGORICAL_FEATURES)
    X = pd.concat([X, cats], axis=1)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0)
    return X

X_train_raw = build_features(train_df)
X_test_raw = build_features(test_df, columns=X_train_raw.columns)
print(X_train_raw.shape, X_test_raw.shape)

banned columns touched: set()
(24000, 61) (6000, 61)


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

In [6]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train)
    scores[k] = silhouette_score(X_train, km.labels_)
    print(f"k={k}: silhouette={scores[k]:.3f}")

best_k = max(scores, key=scores.get)
print("best k:", best_k)

k=2: silhouette=0.338


k=3: silhouette=0.169


k=4: silhouette=0.175


k=5: silhouette=0.169


k=6: silhouette=0.170


k=7: silhouette=0.166


k=8: silhouette=0.151
best k: 2


In [7]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_train)
train_df["cluster"] = km.labels_
test_df["cluster"] = km.predict(X_test)

cluster_rates = train_df.groupby("cluster")["is_declining_label"].mean().sort_values(ascending=False)
print(cluster_rates)

# model score for ranking: this cluster's observed decline rate, learned on train only
test_df["model_score"] = test_df["cluster"].map(cluster_rates)

cluster
0    0.562344
1    0.300885
Name: is_declining_label, dtype: float64


In [8]:
# same rule as the Week-4 baseline (stale_ctr_gap), recomputed on this split
# expected_ctr comes from TRAIN only, so the held-out test comparison stays honest
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

expected_ctr = (
    train_df[(train_df["impressions_90d"] >= 500) & (train_df["position_tier"].isin(tier_order))]
    .groupby("position_tier")["ctr"]
    .median()
)
test_df["expected_ctr"] = test_df["position_tier"].map(expected_ctr)
test_df["ctr_gap"] = (test_df["expected_ctr"] - test_df["ctr"]).clip(lower=0)

stale = (test_df["days_since_last_update"] >= 90).astype(int)
is_visible = (test_df["impressions_90d"] >= 500).astype(int)
has_position = test_df["position_tier"].isin(tier_order).astype(int)

test_df["baseline_score"] = stale * is_visible * has_position * test_df["ctr_gap"] * test_df["impressions_90d"]
print("flagged on test (score > 0):", (test_df["baseline_score"] > 0).sum(), "of", len(test_df))

flagged on test (score > 0): 640 of 6000


In [9]:
def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

base_rate = test_df["is_declining_label"].mean()

rows = []
for k in (25, 50, 100):
    rows.append({
        "k": k,
        "baseline_precision": precision_at_k(test_df["is_declining_label"].values, test_df["baseline_score"].values, k),
        "model_precision": precision_at_k(test_df["is_declining_label"].values, test_df["model_score"].values, k),
        "base_rate": base_rate,
    })

compare = pd.DataFrame(rows)
compare

,k,baseline_precision,model_precision,base_rate
0,25,0.48,0.52,0.544667
1,50,0.56,0.64,0.544667
2,100,0.59,0.60,0.544667


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
profile_cols = ["impressions_90d", "avg_position", "ctr", "days_since_last_update",
                "engagement_rate", "is_declining_label"]
profile = train_df.groupby("cluster")[profile_cols].mean().round(2)
profile["n"] = train_df.groupby("cluster").size()
profile

,impressions_90d,avg_position,ctr,days_since_last_update,engagement_rate,is_declining_label,n
cluster,,,,,,,
0,5607.84,17.14,0.32,47.30,2.58,0.56,22079
1,345.90,7.21,2.70,35.09,2.03,0.30,1921


**Cluster 0 -- "high-traffic underperformers" (22,079 pages, 92%).** Big impression counts
(mean 5,608), worse average position (17.1, striking / page_3_5 territory), low ctr (0.32), and
the higher decline rate of the two clusters (56%). Action: improve / review_for_refresh.

**Cluster 1 -- "low-volume steady pages" (1,921 pages, 8%).** Far fewer impressions (346), much
better position (7.2, close to page 1), high ctr (2.70), lower decline rate (30%). These pages
are working fine at a small scale. Action: monitor.

Silhouette picked k=2 by a wide margin (0.338 vs roughly 0.17 for every other k I tried). That's
a coarser split than I went in expecting -- basically "big pages that are struggling" vs "small
pages that are fine" -- but it's the split the data actually supports, not the one I was hoping
to find.

In [11]:
# concrete cases: pages the model scores as high-risk that aren't actually declining
worst_model_misses = (
    test_df.sort_values("model_score", ascending=False)
    .query("is_declining_label == 0")
    .head(3)
)
worst_model_misses[["content_id", "client_id", "cluster", "model_score",
                     "avg_position", "ctr", "days_since_last_update", "trend_pct"]]

,content_id,client_id,cluster,model_score,avg_position,ctr,days_since_last_update,trend_pct
13981,content_22d5b2df365b,client_19581e27de,0,0.562344,2.4,0.58,104,-11.8
17679,content_3cdfcc3b7de0,client_19581e27de,0,0.562344,4.7,0.54,104,-19.5
26515,content_68c5e7d6c717,client_a88a7902cb,0,0.562344,5.4,0.65,20,80.3


All three worst model misses are cluster-0 pages (the higher-decline-rate cluster) that turned
out not to be declining. Two of them (`content_22d5b2df365b`, `content_3cdfcc3b7de0`) sit at
position 2.4 and 4.7 -- top 3 -- with ctr around 0.5-0.6, pages that look fine by any normal
read; the cluster just also holds their high impression count against them. The third
(`content_68c5e7d6c717`) actually rose 80.3% over the last 30 days, the opposite of declining --
the cluster gave it a mediocre score based on traffic volume alone, with no read on which
direction the page was actually moving.

That's the real weakness of scoring by cluster average: every page in cluster 0 gets the exact
same 0.562 score no matter how it's individually doing, so at k=100 the model is basically
guessing among 22,079 tied pages. It wins clearly at k=25 and k=50 (0.52 vs 0.48, 0.64 vs 0.56)
because cluster membership alone is doing real work at that scale, but that edge won't hold as k
gets close to cluster size.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.